In [ ]:
import ROOT
%load_ext JupyROOT

In [ ]:
%%cpp
const float lambda = 1.;
const long long NENTRIES = 100000;

In [ ]:
%%cpp
    gStyle->SetOptStat(0);
    TRandom3 *rand = new TRandom3(0);

    TF1 *ExpCDF = new TF1("ExpCDF", "1-exp(-[0]*x)", 0., 5.);
    TF1 *ExpPDF = new TF1("ExpPDF", "[0]*exp(-[0]*x)", 0., 5.);

    ExpCDF->SetParameter(0, lambda);
    ExpPDF->SetParameter(0, lambda);

    TF1 *ExpInvCDF = new TF1("ExpInvCDF", "-(1./[0])*log(x)", 0., 5);
    ExpInvCDF->SetParameter(0, lambda);

    TH1F *MCExp = new TH1F("MCExp", "", 40, 0., 5.);

    TF1 *ExpPDF_fit = new TF1("ExpPDF_fit", "[1]*exp(-[0]*x)", 0., 5.);
    ExpPDF_fit->SetParameter(0, lambda);
    ExpPDF_fit->SetParameter(1, NENTRIES);

    TCanvas *cExp = new TCanvas("cExp", "cExp", 1100, 550);
    cExp->Divide(2,1);
    cExp->cd(1);
    ExpPDF->SetLineColor(kBlue);
    ExpPDF->SetLineWidth(2);
    ExpPDF->SetTitle("PDF and CDF;x;y");
    ExpPDF->Draw();
    ExpCDF->SetLineColor(kRed);
    ExpCDF->SetLineWidth(2);
    ExpCDF->GetYaxis()->SetRangeUser(0.,1.);
    ExpCDF->Draw("same");

    for(int i = 0; i < NENTRIES; i++)
    {
        float u = rand->Uniform(0., 1.);

        float x = ExpInvCDF->Eval(u);

        MCExp->Fill(x);
    }

    cExp->cd(2);
    //MCExp->Scale(1./MCExp->Integral());
    MCExp->Fit(ExpPDF_fit, "L", "", 0., 5.);
    MCExp->SetTitle("MC generated distribution;x;Entries");
    MCExp->Draw();
    TLatex *text = new TLatex(0.3, 0.8, Form("#lambda from fit = %.3lf(%.3lf)", ExpPDF_fit->GetParameter(0), ExpPDF_fit->GetParError(0)));
    text->SetNDC();
    text->SetTextSize(0.05);
    text->SetTextColor(kBlack);
    text->Draw("same");